# Clickbait Spoiler Generation

In this notebook we build a small spoiler-generation prototype for clickbait headlines. We use the Webis-Clickbait-22 dataset and `llama3:latest` through the WU Ollama endpoint.

The goal is simple. We load the data, prepare short article contexts, generate spoiler variants, and export the results for later checking.


## Notebook flow

First we load the dataset, then we prepare the rows, set up the LLM connection, define the prompts, and run generation.

This notebook does not evaluate the spoilers. Evaluation belongs in the separate evaluation notebook.


## Data loading

We start with the three predefined dataset splits. The important fields are the clickbait post, article paragraphs, gold spoiler, and spoiler type.


In [1]:
from pathlib import Path
import os
import re

import pandas as pd
import requests
from IPython.display import display

# Make long text columns easier to inspect in notebook tables.
pd.set_option("display.max_colwidth", 120)

# Dataset files from Webis-Clickbait-22.
DATA_DIR = Path("data")
SPLIT_FILES = {
    "train": DATA_DIR / "train.jsonl",
    "validation": DATA_DIR / "validation.jsonl",
    "test": DATA_DIR / "test.jsonl",
}
EXPECTED_SPLIT_SIZES = {"train": 3_200, "validation": 800, "test": 1_000}

# We send only a shortened article context to the model.
MAX_CONTEXT_CHARS = 3_000


## Helper functions

The dataset stores some fields as lists. These helpers turn them into plain strings and create a simple first-sentence baseline.


In [2]:
def load_jsonl(path):
    """Load one JSONL dataset split."""
    return pd.read_json(path, lines=True, encoding="utf-8")


def join_text(value):
    """Convert list-valued dataset fields into normal strings."""
    if isinstance(value, list):
        return " ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)


def first_sentence(article_context):
    """Use the first article sentence as a simple baseline spoiler."""
    text = join_text(article_context).strip()
    if not text:
        return ""
    return re.split(r"(?<=[.!?])\s+", text, maxsplit=1)[0].strip()


## Load and check the data

Before generation, we check that all expected files exist, the split sizes match the README, and the required fields are present.


In [3]:
# Check that the local dataset files are available.
missing_files = [str(path) for path in SPLIT_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing dataset files: {missing_files}")

# Load all splits into memory.
splits = {name: load_jsonl(path) for name, path in SPLIT_FILES.items()}

# Confirm that the local files match the expected split sizes.
actual_split_sizes = {name: len(frame) for name, frame in splits.items()}
assert actual_split_sizes == EXPECTED_SPLIT_SIZES, actual_split_sizes

# Confirm that all splits have the same schema and the fields we need.
required_fields = {"uuid", "postText", "targetParagraphs", "spoiler", "tags"}
reference_columns = list(splits["train"].columns)
for split_name, frame in splits.items():
    missing_fields = required_fields.difference(frame.columns)
    assert not missing_fields, f"{split_name} is missing fields: {sorted(missing_fields)}"
    assert list(frame.columns) == reference_columns, f"Schema mismatch in {split_name}"

# Show a compact summary for presentation.
dataset_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(frame),
            "columns": len(frame.columns),
            "duplicate_ids": int(frame["uuid"].duplicated().sum()),
        }
        for split_name, frame in splits.items()
    ]
)
display(dataset_summary)


,split,rows,columns,duplicate_ids
0,train,3200,14,0
1,validation,800,14,0
2,test,1000,14,0


## Data preparation

For generation we create one clean table. It contains the row id, split name, clickbait text, spoiler type, shortened article context, and gold spoiler.

The generated columns are added later, after the model runs.


In [4]:
# These are the final columns that will be saved to CSV and JSONL.
OUTPUT_COLUMNS = [
    "id",
    "split",
    "post_text",
    "spoiler_type",
    "article_context_short",
    "gold_spoiler",
    "generated_question",
    "first_sentence_baseline",
    "direct_llama_spoiler",
    "question_llama_spoiler",
    "type_aware_llama_spoiler",
]


def prepare_generation_input(run_limit=None):
    """Create the combined generation input table from all dataset splits."""
    prepared_splits = []

    for split_name, frame in splits.items():
        prepared_splits.append(
            pd.DataFrame(
                {
                    "id": frame["uuid"].map(join_text),
                    "split": split_name,
                    "post_text": frame["postText"].map(join_text),
                    "spoiler_type": frame["tags"].map(join_text),
                    "article_context_short": frame["targetParagraphs"].map(join_text).str[:MAX_CONTEXT_CHARS],
                    "gold_spoiler": frame["spoiler"].map(join_text),
                }
            )
        )

    combined = pd.concat(prepared_splits, ignore_index=True)
    if run_limit is not None:
        combined = combined.head(run_limit).copy()
    return combined.reset_index(drop=True)


# Preview a few prepared rows before running the model.
preview_rows = prepare_generation_input(run_limit=5)
display(preview_rows[["id", "split", "post_text", "spoiler_type", "gold_spoiler"]])


,id,split,post_text,spoiler_type,gold_spoiler
0,0af11f6b-c889-4520-9372-66ba25cb7657,train,"Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had Better Idea",passage,how about that morning we go throw?
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,train,NASA sets date for full recovery of ozone hole,phrase,2070
2,008b7b19-0445-4e16-8f9e-075b73f80ca4,train,This is what makes employees happy -- and it's not their paycheck,phrase,intellectual stimulation
3,31ecf93c-3e21-4c80-949b-aa549a046b93,train,Passion is overrated — 7 work habits you need instead,multi,"Purpose connects us to something bigger and in doing so makes us right sized be ruthless with your ""No’s."" Practice ..."
4,31b108a3-c828-421a-a4b9-cf651e9ac859,train,The perfect way to cook rice so that it's perfectly fluffy and NEVER sticks to the pan,phrase,in a rice cooker


## Ollama setup

We use the WU Ollama-compatible `/api/generate` endpoint. If the request fails, we should first check the WU network or VPN connection.


In [5]:
# The endpoint can be overridden with an environment variable if needed.
OLLAMA_URL = os.getenv(
    "OLLAMA_URL", "https://ollama-gpt-oss.cluster.ai.wu.ac.at"
).rstrip("/")
OLLAMA_MODEL = "llama3:latest"
REQUEST_TIMEOUT_SECONDS = int(os.getenv("OLLAMA_TIMEOUT", "120"))


class OllamaAPIError(RuntimeError):
    """Readable error for failed Ollama calls."""


def call_llama3(prompt, temperature=0.2, num_predict=100):
    """Send one prompt to llama3:latest and return only the response text."""
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": num_predict,
        },
    }
    url = f"{OLLAMA_URL}/api/generate"

    # Network errors usually mean the endpoint or VPN is not reachable.
    try:
        response = requests.post(url, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
    except requests.RequestException as exc:
        raise OllamaAPIError(
            f"Could not connect to {url}. Check the WU network/VPN. Original error: {exc}"
        ) from exc

    # Return clearer messages for common API failures.
    if response.status_code == 403:
        raise OllamaAPIError("The WU Ollama server returned 403 Forbidden. Check the campus/VPN connection.")
    if not response.ok:
        raise OllamaAPIError(f"Ollama request failed with {response.status_code}: {response.text[:500]}")

    try:
        data = response.json()
    except ValueError as exc:
        raise OllamaAPIError(f"Ollama returned non-JSON content: {response.text[:500]}") from exc

    if data.get("error"):
        raise OllamaAPIError(str(data["error"]))
    if "response" not in data:
        raise OllamaAPIError(f"Unexpected Ollama response: {data}")

    return data["response"].strip()


print(f"Ollama endpoint: {OLLAMA_URL}")
print(f"Model: {OLLAMA_MODEL}")


Ollama endpoint: https://ollama-gpt-oss.cluster.ai.wu.ac.at
Model: llama3:latest


## Prompt functions

We use three generation styles. The direct method asks for a spoiler immediately. The question method first rewrites the headline as a question. The type-aware method also tells the model whether the spoiler should be a phrase, passage, or multipart answer.

All prompts ask the model to use only the article context and keep the answer short.


In [6]:
# Type-specific formatting rules for the spoiler prompt.
SPOILER_TYPE_RULES = {
    "phrase": "Return a short phrase.",
    "passage": "Return one short sentence.",
    "multipart": "Return a short semicolon-separated list.",
}


def convert_clickbait_to_question(post_text):
    """Rewrite a clickbait headline as one short question."""
    post_text = join_text(post_text).strip()
    prompt = f"""
Rewrite the clickbait post below as one short, direct question that captures its curiosity gap.
Do not answer the question. Do not add facts or assumptions.
Return only the question, with no explanation or quotation marks.

Clickbait post:
{post_text}
""".strip()
    return call_llama3(prompt, temperature=0.2, num_predict=60)


def generate_direct_spoiler(post_text, article_context):
    """Generate a spoiler directly from the headline and article context."""
    prompt = f"""
Write a concise spoiler that resolves the clickbait post.
Use only facts explicitly supported by the article context. Do not invent or infer facts.
Keep the spoiler short. Return only the spoiler, with no explanation, label, or quotation marks.

Clickbait post:
{join_text(post_text).strip()}

Article context:
{join_text(article_context).strip()}
""".strip()
    return call_llama3(prompt, temperature=0.2, num_predict=100)


def generate_spoiler(post_text, generated_question, article_context, spoiler_type=None):
    """Generate a spoiler, optionally using the dataset spoiler type."""
    normalized_type = join_text(spoiler_type).strip().lower() if spoiler_type else None
    if normalized_type == "multi":
        normalized_type = "multipart"
    if normalized_type and normalized_type not in SPOILER_TYPE_RULES:
        raise ValueError(f"Unknown spoiler_type {spoiler_type!r}; expected phrase, passage, or multipart.")

    type_rule = SPOILER_TYPE_RULES.get(
        normalized_type, "Return the shortest answer that resolves the clickbait."
    )
    prompt = f"""
Write a concise spoiler that resolves the clickbait post and answers the generated question.
Use only facts explicitly supported by the article context. Do not invent or infer facts.
{type_rule}
Return only the spoiler, with no explanation, label, or quotation marks.

Clickbait post:
{join_text(post_text).strip()}

Generated question:
{join_text(generated_question).strip()}

Article context:
{join_text(article_context).strip()}
""".strip()
    return call_llama3(prompt, temperature=0.2, num_predict=100)


## Generation settings

This is the main place to change the run size. `RUN_LIMIT = 300` means we process the first 300 examples. Use `None` if we want the full dataset.

The notebook saves to the `output` folder.


In [ ]:
OUTPUT_DIR = "output"
RUN_LIMIT = 10
SAVE_EVERY = 10

if RUN_LIMIT is not None and RUN_LIMIT <= 0:
    raise ValueError("RUN_LIMIT must be None or a positive integer.")
if SAVE_EVERY <= 0:
    raise ValueError("SAVE_EVERY must be a positive integer.")

# Set output paths and prepare the rows for this run.
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
csv_output_path = output_dir / "spoiler_generation_results.csv"
jsonl_output_path = output_dir / "spoiler_generation_results.jsonl"

generation_input = prepare_generation_input(run_limit=RUN_LIMIT)
order_lookup = {
    (row_id, split_name): position
    for position, (row_id, split_name) in enumerate(zip(generation_input["id"], generation_input["split"]))
}

print(f"Rows requested: {len(generation_input)}")
print(f"CSV output: {csv_output_path}")
print(f"JSONL output: {jsonl_output_path}")


Rows requested: 10
CSV output: output\spoiler_generation_results.csv
JSONL output: output\spoiler_generation_results.jsonl


## Saving and checkpoints

Generation can take time, so we save intermediate results every few rows. If the notebook stops, we can run it again and it will skip completed rows.

Rows with missing generated text are retried instead of treated as finished.


In [8]:
def order_results(results):
    """Keep saved rows in the same order as the current generation input."""
    ordered = results.drop_duplicates(subset=["id", "split"], keep="last").copy()
    ordered["_order"] = [
        order_lookup.get((row_id, split_name), len(order_lookup))
        for row_id, split_name in zip(ordered["id"], ordered["split"])
    ]
    return ordered.sort_values("_order").drop(columns="_order").reset_index(drop=True)


def save_generation_outputs(results):
    """Save the current results to both CSV and JSONL."""
    ordered_results = order_results(results)[OUTPUT_COLUMNS]
    csv_temp_path = csv_output_path.with_suffix(csv_output_path.suffix + ".tmp")
    jsonl_temp_path = jsonl_output_path.with_suffix(jsonl_output_path.suffix + ".tmp")

    # Write temporary files first, then replace the final files.
    ordered_results.to_csv(csv_temp_path, index=False, encoding="utf-8")
    ordered_results.to_json(jsonl_temp_path, orient="records", lines=True, force_ascii=False)
    csv_temp_path.replace(csv_output_path)
    jsonl_temp_path.replace(jsonl_output_path)


def safe_generation_call(label, function, *args, **kwargs):
    """Keep the run going if one LLM call fails."""
    try:
        return function(*args, **kwargs)
    except Exception as exc:
        print(f"  {label} failed: {exc}")
        return ""


## Run the pipeline

For each row, we create the first-sentence baseline and three LLM outputs. The final table keeps the same columns for every method, so it is easy to inspect or evaluate later.


In [9]:
# Load completed rows if a checkpoint already exists.
if csv_output_path.exists():
    output_results = pd.read_csv(csv_output_path, dtype=str, keep_default_na=False, encoding="utf-8")
    if list(output_results.columns) != OUTPUT_COLUMNS:
        raise ValueError(f"Existing checkpoint has unexpected columns: {list(output_results.columns)}")

    # Keep only rows that belong to the current requested run.
    requested_keys = set(order_lookup)
    output_results = output_results[
        [(row_id, split_name) in requested_keys for row_id, split_name in zip(output_results["id"], output_results["split"])]
    ].copy()

    # Retry rows where one or more LLM generations are still empty.
    generated_columns = [
        "generated_question",
        "direct_llama_spoiler",
        "question_llama_spoiler",
        "type_aware_llama_spoiler",
    ]
    complete_mask = output_results[generated_columns].fillna("").apply(
        lambda row: all(str(value).strip() for value in row), axis=1
    )
    incomplete_count = int((~complete_mask).sum())
    if incomplete_count:
        print(f"Retrying {incomplete_count} incomplete checkpoint rows.")
    output_results = output_results[complete_mask].reset_index(drop=True)
    print(f"Loaded {len(output_results)} completed examples from {csv_output_path}.")
else:
    output_results = pd.DataFrame(columns=OUTPUT_COLUMNS)

completed_keys = set(zip(output_results["id"], output_results["split"]))
processed_this_run = 0
total_requested = len(generation_input)

# Generate missing rows one by one.
for row_number, row in generation_input.iterrows():
    key = (row["id"], row["split"])
    if key in completed_keys:
        print(f"[{row_number + 1}/{total_requested}] already complete: {row['id']}")
        continue

    print(f"[{row_number + 1}/{total_requested}] generating {row['split']}: {row['id']}")
    generated_question = safe_generation_call(
        "question conversion", convert_clickbait_to_question, row["post_text"]
    )
    direct_spoiler = safe_generation_call(
        "direct spoiler", generate_direct_spoiler, row["post_text"], row["article_context_short"]
    )
    question_spoiler = safe_generation_call(
        "question spoiler", generate_spoiler, row["post_text"], generated_question, row["article_context_short"]
    )
    type_aware_spoiler = safe_generation_call(
        "type-aware spoiler",
        generate_spoiler,
        row["post_text"],
        generated_question,
        row["article_context_short"],
        spoiler_type=row["spoiler_type"],
    )

    output_results.loc[len(output_results)] = {
        "id": row["id"],
        "split": row["split"],
        "post_text": row["post_text"],
        "spoiler_type": row["spoiler_type"],
        "article_context_short": row["article_context_short"],
        "gold_spoiler": row["gold_spoiler"],
        "generated_question": generated_question,
        "first_sentence_baseline": first_sentence(row["article_context_short"]),
        "direct_llama_spoiler": direct_spoiler,
        "question_llama_spoiler": question_spoiler,
        "type_aware_llama_spoiler": type_aware_spoiler,
    }
    completed_keys.add(key)
    processed_this_run += 1

    if processed_this_run % SAVE_EVERY == 0:
        save_generation_outputs(output_results)
        print(f"  Checkpoint saved after {processed_this_run} new examples.")

# Save the final version in both formats.
output_results = order_results(output_results)[OUTPUT_COLUMNS]
save_generation_outputs(output_results)

print(f"Processed examples in this run: {processed_this_run}")
print(f"Total saved examples: {len(output_results)}")
print(f"Output CSV path: {csv_output_path}")
print(f"Output JSONL path: {jsonl_output_path}")
display(output_results.head())


Loaded 10 completed examples from output\spoiler_generation_results.csv.
[1/10] already complete: 0af11f6b-c889-4520-9372-66ba25cb7657
[2/10] already complete: b1a1f63d-8853-4a11-89e8-6b2952a393ec
[3/10] already complete: 008b7b19-0445-4e16-8f9e-075b73f80ca4
[4/10] already complete: 31ecf93c-3e21-4c80-949b-aa549a046b93
[5/10] already complete: 31b108a3-c828-421a-a4b9-cf651e9ac859
[6/10] already complete: 12e3034e-a98f-4cd2-8773-3f7974161c45
[7/10] already complete: 8acc9f97-846a-4c00-9483-82ddd638917d
[8/10] already complete: c4e0edb3-16f0-49eb-b3a5-cdd24c546c74
[9/10] already complete: 6db35eb1-a186-4fb2-a0b8-7b94d0396589
[10/10] already complete: e26fa6ed-e364-4666-af8e-20fbada53839
Processed examples in this run: 0
Total saved examples: 10
Output CSV path: output\spoiler_generation_results.csv
Output JSONL path: output\spoiler_generation_results.jsonl


,id,split,post_text,spoiler_type,article_context_short,gold_spoiler,generated_question,first_sentence_baseline,direct_llama_spoiler,question_llama_spoiler,type_aware_llama_spoiler
0,0af11f6b-c889-4520-9372-66ba25cb7657,train,"Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had Better Idea",passage,It’ll be just like old times this weekend for Tom Brady and Wes Welker. Welker revealed Friday morning on a Miami ra...,how about that morning we go throw?,Did Wes Welker really get rejected by Tom Brady?,It’ll be just like old times this weekend for Tom Brady and Wes Welker.,Tom Brady asked Wes Welker to go throw footballs instead of having dinner.,Tom Brady had a different idea than dinner and instead invited Wes Welker to throw footballs with him.,Tom Brady had no intention of having dinner with Wes Welker and instead invited him to throw passes together.
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,train,NASA sets date for full recovery of ozone hole,phrase,2070 is shaping up to be a great year for Mother Earth. That's when NASA scientists are predicting the hole in the o...,2070,Is NASA finally fixing the ozone hole?,2070 is shaping up to be a great year for Mother Earth.,By 2070.,By 2070.,By 2070.
2,008b7b19-0445-4e16-8f9e-075b73f80ca4,train,This is what makes employees happy -- and it's not their paycheck,phrase,"Despite common belief, money isn't the key to employee happiness, new research finds. A study by hiring software pro...",intellectual stimulation,What makes employees happy?,"Despite common belief, money isn't the key to employee happiness, new research finds.","Intellectual stimulation accounts for 18.5% of job satisfaction, while money only accounts for 5.4%.",Intellectual stimulation.,Intellectual stimulation.
3,31ecf93c-3e21-4c80-949b-aa549a046b93,train,Passion is overrated — 7 work habits you need instead,multi,"It’s common wisdom. Near gospel really, and not just among young people and founders. Across generational lines, sen...","Purpose connects us to something bigger and in doing so makes us right sized be ruthless with your ""No’s."" Practice ...",Is passion really necessary for success?,It’s common wisdom.,Purpose deemphasizes the I and focuses on pursuing something outside yourself.,Not necessary.,Is passion really necessary for success?; No; Passion is overrated; Purpose deemphasizes the I; Purpose is about pur...
4,31b108a3-c828-421a-a4b9-cf651e9ac859,train,The perfect way to cook rice so that it's perfectly fluffy and NEVER sticks to the pan,phrase,"Boiling rice may seem simple, but there is a very fine line between under-cooked crunchy grains and mushy over-cooke...",in a rice cooker,Is there a secret to cooking rice that prevents it from sticking to the pan?,"Boiling rice may seem simple, but there is a very fine line between under-cooked crunchy grains and mushy over-cooke...",One piece of equipment is required: a rice cooker.,"Use a rice cooker or boil rice with a 1:1.5 ratio of rice to water, then simmer for 8 minutes and let rest for 15 mi...",One piece of equipment.


## Demo

This small demo lets us try one custom clickbait headline after the setup cells have run.

#### example: 
https://www.96fm.ie/news/entertainment/shawn-mendes-and-camila-cabello-announce-new-addition-to-the-family/

#### Headline:
"Shawn Mendes and Camila Cabello announce new addition to the family"

#### Article text:
"Shawn Mendes and Camila Cabello have welcomed a new addition to the family.

Celebrity couple Shawn Mendes and Camila Cabello are no strangers to sharing the love on Instagram. 

For anyone who follows the pair on the social media platform, you'll be well used to seeing them cute and in love on your feed, but their Instagram content is about to get a whole lot cuter. 

The couple, who've been dating since summer 2019, have just announced a new addition to the mix - a puppy! 

Shawn and Camila have reached a huge relationship milestone by becoming dog parents. 

First to announce the news on Instagram, Shawn shared a selection of adorable photos and video clips of their new family member, Tarzan the Golden Labrador puppy. 

Tarzan is just the latest addition to the couple's pet family. Camila has three other dogs - Shih Tzu Leo, Chihuahua Eugene and German Shepherd  Thunder.

Sharing the exciting news with her fans, Camila said: "During uncertain times like this we need a reminder that sweet miracle things like puppies exist in the world too, meet the new member of the pack: Tarzan!

"Sending to love all of you guys and remember: regardless of the outcome, WE are the ones responsible for building the world we want to live in. The fight for BEING the society we want to see. That continues after this outcome is decided.

"This is what I’m telling myself to soothe myself right now because it’s the only thing we actually can control. love you guys and Tarzan sends a big puppy lick."

Welcome to the Mendes-Cabello family, Tarzan! "

#### Spoiler type:
"passage"


In [13]:
post_text = input("Clickbait headline: ").strip()
article_context = input("Article text: ").strip()
spoiler_type = input("Spoiler type (phrase, passage, or multipart): ").strip().lower()

if spoiler_type not in {"phrase", "passage", "multipart"}:
    raise ValueError("Spoiler type must be phrase, passage, or multipart.")

# Reuse the same functions as the main generation pipeline.
generated_question = convert_clickbait_to_question(post_text)
generated_spoiler = generate_spoiler(
    post_text,
    generated_question,
    article_context,
    spoiler_type=spoiler_type,
)

print("\nGenerated question:")
print(generated_question)

print("\nGenerated spoiler:")
print(generated_spoiler)



Generated question:
Is Shawn Mendes having a baby?

Generated spoiler:
Shawn Mendes and Camila Cabello are having a puppy.
